# Construction de la couche GOLD

Cette étape transforme les données nettoyées issues de la couche Silver en référentiels structurés utilisés par le moteur d’intelligence artificielle.

Objectifs :

- Construire un référentiel de compétences
- Construire un référentiel métiers
- Construire une table de correspondance métier-compétences

Ces données serviront ensuite à l’analyse sémantique et au moteur de recommandation.

In [1]:
import pandas as pd

# Chargement du dataset nettoyé

Nous récupérons les données standardisées depuis la couche Silver.

In [2]:
df = pd.read_csv("../data/silver/cleaned_jobs.csv")

In [3]:
print(df.shape)

(200, 13)


# Extraction des compétences

Nous analysons la colonne skills afin d’extraire toutes les compétences techniques présentes dans le dataset.

In [4]:
all_skills = []

for skill_list in df["skills"]:
    
    skills = skill_list.split(",")

    for skill in skills:
        all_skills.append(skill.strip())

In [5]:
unique_skills = sorted(list(set(all_skills)))

print(unique_skills)

['AWS', 'Agile', 'CI/CD', 'Docker', 'Git', 'Go', 'Java', 'JavaScript', 'Kubernetes', 'Machine Learning', 'MongoDB', 'Node.js', 'Python', 'REST APIs', 'React', 'Ruby', 'SQL', 'TensorFlow', 'TypeScript']


# Construction des blocs de compétences

Afin de permettre un scoring par domaine, les compétences sont regroupées en blocs cohérents.

In [6]:
competency_mapping = {

    "AWS": "Cloud & DevOps",
    "Docker": "Cloud & DevOps",
    "Kubernetes": "Cloud & DevOps",
    "CI/CD": "Cloud & DevOps",
    "Git": "Cloud & DevOps",

    "Python": "Programming",
    "Java": "Programming",
    "JavaScript": "Programming",
    "TypeScript": "Programming",
    "Ruby": "Programming",
    "Go": "Programming",
    "Node.js": "Programming",

    "Machine Learning": "AI / Machine Learning",
    "TensorFlow": "AI / Machine Learning",

    "REST APIs": "Backend Engineering",
    "React": "Backend Engineering",

    "SQL": "Data Engineering",
    "MongoDB": "Data Engineering",

    "Agile": "Software Engineering Practices"
}

In [7]:
competencies_df = pd.DataFrame({

    "competency_id": [f"C{i+1:03}" for i in range(len(unique_skills))],

    "competency": unique_skills
})

competencies_df["block"] = competencies_df["competency"].map(
    competency_mapping
)

competencies_df

,competency_id,competency,block
0,C001,AWS,Cloud & DevOps
1,C002,Agile,Software Engineering Practices
2,C003,CI/CD,Cloud & DevOps
3,C004,Docker,Cloud & DevOps
4,C005,Git,Cloud & DevOps
5,C006,Go,Programming
6,C007,Java,Programming
7,C008,JavaScript,Programming
8,C009,Kubernetes,Cloud & DevOps
9,C010,Machine Learning,AI / Machine Learning


In [8]:
semantic_descriptions = {
    "AWS": "Expérience dans le déploiement d'infrastructures cloud et d'applications avec Amazon Web Services.",
    "Agile": "Expérience du travail en équipe agile et de la gestion de projet par itérations.",
    "CI/CD": "Capacité à automatiser les processus de compilation, de test et de déploiement logiciel avec des pipelines CI/CD.",
    "Docker": "Capacité à conteneuriser des applications et à gérer des environnements reproductibles avec Docker.",
    "Git": "Capacité à versionner, gérer et collaborer sur du code source avec Git.",
    "Go": "Capacité à développer des services backend et des applications avec le langage Go.",
    "Java": "Capacité à développer des applications backend ou d'entreprise avec Java.",
    "JavaScript": "Capacité à développer des applications web interactives avec JavaScript.",
    "Kubernetes": "Capacité à orchestrer et déployer des applications conteneurisées avec Kubernetes.",
    "Machine Learning": "Capacité à entraîner, évaluer et déployer des modèles de machine learning pour des tâches prédictives.",
    "MongoDB": "Capacité à stocker, interroger et gérer des données NoSQL avec MongoDB.",
    "Node.js": "Capacité à développer des applications serveur et des API avec Node.js.",
    "Python": "Capacité à développer des logiciels, scripts d'automatisation et applications data avec Python.",
    "REST APIs": "Capacité à concevoir, consommer et documenter des API REST pour des systèmes logiciels.",
    "React": "Capacité à développer des interfaces utilisateur basées sur des composants avec React.",
    "Ruby": "Capacité à développer des applications web et services backend avec Ruby.",
    "SQL": "Capacité à interroger, manipuler et analyser des données structurées avec des bases SQL.",
    "TensorFlow": "Capacité à construire et entraîner des modèles de deep learning avec TensorFlow.",
    "TypeScript": "Capacité à développer des applications web robustes et typées avec TypeScript."
}

competencies_df["semantic_description"] = competencies_df["competency"].map(
    semantic_descriptions
)

competencies_df

,competency_id,competency,block,semantic_description
0,C001,AWS,Cloud & DevOps,Expérience dans le déploiement d'infrastructur...
1,C002,Agile,Software Engineering Practices,Expérience du travail en équipe agile et de la...
2,C003,CI/CD,Cloud & DevOps,Capacité à automatiser les processus de compil...
3,C004,Docker,Cloud & DevOps,Capacité à conteneuriser des applications et à...
4,C005,Git,Cloud & DevOps,"Capacité à versionner, gérer et collaborer sur..."
5,C006,Go,Programming,Capacité à développer des services backend et ...
6,C007,Java,Programming,Capacité à développer des applications backend...
7,C008,JavaScript,Programming,Capacité à développer des applications web int...
8,C009,Kubernetes,Cloud & DevOps,Capacité à orchestrer et déployer des applicat...
9,C010,Machine Learning,AI / Machine Learning,"Capacité à entraîner, évaluer et déployer des ..."


In [9]:
competencies_df.to_csv(
    "../data/gold/competencies.csv",
    index=False
)

# Construction du référentiel métiers

Nous extrayons l’ensemble des métiers présents dans le dataset.

In [10]:
jobs_df = df[["job_title", "category"]]

jobs_df = jobs_df.drop_duplicates()

jobs_df = jobs_df.reset_index(drop=True)

In [11]:
jobs_df["job_id"] = [
    f"J{i+1:03}" for i in range(len(jobs_df))
]

In [12]:
jobs_df = jobs_df[
    ["job_id", "job_title", "category"]
]

jobs_df.head()

,job_id,job_title,category
0,J001,Engineering Manager,Technology
1,J002,Lead Engineer,Technology
2,J003,Senior Software Engineer,Technology
3,J004,Senior Data Scientist,Technology
4,J005,Project Manager,Technology


In [13]:
jobs_df.to_csv(
    "../data/gold/jobs.csv",
    index=False
)

# Construction des relations métiers-compétences

Chaque métier est relié aux compétences présentes dans la colonne skills.

In [14]:
competency_dict = dict(
    zip(
        competencies_df["competency"],
        competencies_df["competency_id"]
    )
)

In [15]:
mapping_rows = []

for _, row in df.iterrows():

    job_match = jobs_df[
        jobs_df["job_title"] == row["job_title"]
    ]

    job_id = job_match.iloc[0]["job_id"]

    skills = [
        skill.strip()
        for skill in row["skills"].split(",")
    ]

    for skill in skills:

        if skill in competency_dict:

            mapping_rows.append({

                "job_id": job_id,

                "competency_id": competency_dict[skill]
            })

In [16]:
mapping_df = pd.DataFrame(mapping_rows)

mapping_df = mapping_df.drop_duplicates()

In [17]:
mapping_df.to_csv(
    "../data/gold/mapping.csv",
    index=False
)

In [18]:
print(competencies_df.shape)

print(jobs_df.shape)

print(mapping_df.shape)

(19, 4)
(22, 3)
(385, 2)


In [20]:
print(jobs_df["job_title"].unique())

<StringArray>
[      'Engineering Manager',             'Lead Engineer',
  'Senior Software Engineer',     'Senior Data Scientist',
           'Project Manager',           'Product Manager',
            'Technical Lead',            'Data Scientist',
        'Frontend Developer',       'Solutions Architect',
         'Software Engineer',         'Backend Developer',
               'QA Engineer', 'Machine Learning Engineer',
           'DevOps Engineer',         'Security Engineer',
      'Full Stack Developer',               'UX Designer',
          'Junior Developer',      'System Administrator',
          'Business Analyst',              'Data Analyst']
Length: 22, dtype: str


# Conclusion de la couche GOLD

La couche Gold contient désormais :

- Un référentiel de compétences structuré
- Un référentiel métiers
- Une table de relations entre métiers et compétences

Ces fichiers seront utilisés par le moteur d’analyse sémantique afin d’effectuer :

- vectorisation des compétences
- comparaison sémantique
- scoring
- recommandation métier